# S6E5 — XGBoost Model

Pit stop prediction using the feature engineering roadmap from `docs/eda_insights.md`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from pathlib import Path
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import KFold

# Compound tyre life thresholds from EDA (Finding 2.1)
COMPOUND_MEDIAN_PIT_LIFE = {
    "SOFT": 12, "MEDIUM": 16, "HARD": 20, "INTERMEDIATE": 17, "WET": 11,
}
COMPOUND_Q75_PIT_LIFE = {
    "SOFT": 16, "MEDIUM": 22, "HARD": 27, "INTERMEDIATE": 24, "WET": 17,
}

In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^a-z0-9_]", "", regex=True)
    )
    return df


def add_static_features(df: pd.DataFrame) -> pd.DataFrame:
    """Features that require no target-level information from training data."""
    df = df.copy()
    rp = df["raceprogress"]
    compound = df["compound"]

    # Priority 1 — tyre life
    median_life = compound.map(COMPOUND_MEDIAN_PIT_LIFE).fillna(15).clip(lower=1)
    q75_life = compound.map(COMPOUND_Q75_PIT_LIFE).fillna(20)
    df["tyre_life_ratio"] = df["tyrelife"] / median_life
    df["tyre_life_over_cliff"] = (df["tyrelife"] > q75_life).astype(int)
    df["tyre_life_remaining_to_cliff"] = q75_life - df["tyrelife"]

    # Priority 1 — race phase / pit windows
    df["race_phase"] = np.select(
        [rp < 0.05, rp < 0.35, rp < 0.65, rp < 0.85],
        [0, 1, 2, 3],
        default=4,
    )
    df["in_pit_window"] = (
        ((rp >= 0.25) & (rp <= 0.45)) | ((rp >= 0.52) & (rp <= 0.72))
    ).astype(int)
    df["too_late_to_pit"] = (rp > 0.85).astype(int)
    df["closing_lap_flag"] = (rp > 0.90).astype(int)

    # Priority 1 — stint
    df["is_likely_final_stint"] = (df["stint"] >= 4).astype(int)
    df["stints_completed"] = df["stint"] - 1

    # Priority 2 — degradation
    df["degradation_rate"] = df["cumulative_degradation"] / df["tyrelife"].clip(lower=1)
    df["tyre_stress"] = df["tyre_life_ratio"] * df["degradation_rate"].abs()

    # Priority 2 — lap time
    df["laptime_delta_clipped"] = df["laptime_delta"].clip(-20, 30)

    # Priority 3 — position
    df["losing_positions"] = (df["position_change"] < -1).astype(int)
    df["position_group"] = pd.cut(
        df["position"].fillna(20),
        bins=[0, 3, 6, 10, 15, 100],
        labels=[0, 1, 2, 3, 4],
    ).astype(int)
    df["undercut_zone"] = (
        (df["position"] > 6) & (rp >= 0.30) & (rp <= 0.70)
    ).astype(int)

    # Priority 3 — pit recency
    df["just_pitted"] = df["pitstop"]

    # Priority 4 — year anomaly flag
    df["year_2023_flag"] = (df["year"] == 2023).astype(int)

    # Priority 1 — compound one-hot (non-ordinal per EDA Finding 2.2)
    for c in ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"]:
        df[f"compound_{c.lower()}"] = (compound == c).astype(int)

    return df


def fit_encodings(df: pd.DataFrame, y: pd.Series) -> dict:
    """Learn target encodings and lookup tables from a training split."""
    global_mean = float(y.mean())

    def target_enc(col, k):
        stats = y.groupby(df[col]).agg(["sum", "count"])
        return ((stats["sum"] + global_mean * k) / (stats["count"] + k)).to_dict()

    laptime_delta_clipped = df["laptime_delta"].clip(-20, 30)
    laptime_q75 = laptime_delta_clipped.groupby(df["compound"]).quantile(0.75).to_dict()

    race_laps = (
        df.groupby(["race", "year"])["lapnumber"]
        .max()
        .reset_index()
        .rename(columns={"lapnumber": "race_laps_total"})
    )

    return {
        "global_mean": global_mean,
        "driver_enc": target_enc("driver", k=20),
        "race_enc": target_enc("race", k=10),
        "year_enc": target_enc("year", k=15),
        "stint_enc": target_enc("stint", k=10),
        "laptime_q75_by_compound": laptime_q75,
        "race_laps": race_laps,
    }


def apply_encodings(df: pd.DataFrame, encodings: dict) -> pd.DataFrame:
    """Apply precomputed encodings — must be called after add_static_features."""
    df = df.copy()
    gm = encodings["global_mean"]

    df["driver_encoded"] = df["driver"].map(encodings["driver_enc"]).fillna(gm)
    df["race_encoded"] = df["race"].map(encodings["race_enc"]).fillna(gm)
    df["year_encoded"] = df["year"].map(encodings["year_enc"]).fillna(gm)
    df["stint_encoded"] = df["stint"].map(encodings["stint_enc"]).fillna(gm)

    laptime_q75 = df["compound"].map(encodings["laptime_q75_by_compound"]).fillna(0)
    df["laptime_delta_above_compound_q75"] = (
        df["laptime_delta_clipped"] > laptime_q75
    ).astype(int)

    fallback_laps = int(df["lapnumber"].max())
    df = df.merge(encodings["race_laps"], on=["race", "year"], how="left")
    df["race_laps_total"] = df["race_laps_total"].fillna(fallback_laps)

    return df


def build_features(df: pd.DataFrame, encodings: dict) -> pd.DataFrame:
    df = add_static_features(df)
    df = apply_encodings(df, encodings)
    return df


FEATURE_COLS = [
    # Raw numeric
    "tyrelife", "lapnumber", "raceprogress", "cumulative_degradation",
    "position", "position_change", "laptime_s", "year",
    # Tyre life
    "tyre_life_ratio", "tyre_life_over_cliff", "tyre_life_remaining_to_cliff", "tyre_stress",
    # Race progress
    "race_phase", "in_pit_window", "too_late_to_pit", "closing_lap_flag",
    # Stint
    "stint", "is_likely_final_stint", "stints_completed",
    # Degradation
    "degradation_rate",
    # Lap time
    "laptime_delta_clipped", "laptime_delta_above_compound_q75",
    # Position
    "position_group", "losing_positions", "undercut_zone",
    # Pit recency
    "just_pitted",
    # Year
    "year_2023_flag",
    # Compound one-hot
    "compound_soft", "compound_medium", "compound_hard", "compound_intermediate", "compound_wet",
    # Target encoded
    "driver_encoded", "race_encoded", "year_encoded", "stint_encoded",
    # Race context
    "race_laps_total",
]

In [ ]:
DATA_DIR = Path("..") / "data"
SUBMISSIONS_DIR = Path("..") / "submissions"

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "scale_pos_weight": 4,  # 4:1 class imbalance (EDA Finding 1.1)
    "n_estimators": 2000,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 30,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "device": "cpu",
    "random_state": 42,
    "n_jobs": -1,
    "early_stopping_rounds": 50,
}


def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(DATA_DIR / "train.csv")
    test = pd.read_csv(DATA_DIR / "test.csv")
    train = clean_columns(train)
    test = clean_columns(test)
    return train, test


def run_cv(train: pd.DataFrame) -> tuple[np.ndarray, list[int]]:
    """5-fold CV. Returns OOF predictions and best iteration per fold."""
    X = train.drop(columns=["id", "pitnextlap"])
    y = train["pitnextlap"]

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(train))
    best_iters: list[int] = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        encodings = fit_encodings(X_tr, y_tr)
        X_tr_feat = build_features(X_tr, encodings)[FEATURE_COLS]
        X_val_feat = build_features(X_val, encodings)[FEATURE_COLS]

        model = xgb.XGBClassifier(**XGB_PARAMS)
        model.fit(
            X_tr_feat, y_tr,
            eval_set=[(X_val_feat, y_val)],
            verbose=200,
        )

        oof_preds[val_idx] = model.predict_proba(X_val_feat)[:, 1]
        best_iters.append(model.best_iteration)
        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        print(f"Fold {fold + 1} | AUC: {fold_auc:.5f} | best_iter: {model.best_iteration}")

    fold_aucs = [
        roc_auc_score(y.iloc[val_idx], oof_preds[val_idx])
        for _, val_idx in KFold(n_splits=5, shuffle=True, random_state=42).split(X)
    ]
    oof_auc = roc_auc_score(y, oof_preds)
    std = np.std(fold_aucs)
    print(f"\nOOF AUC: {oof_auc:.5f} \u00b1 {std:.5f}")
    return oof_preds, best_iters


def train_full_and_predict(
    train: pd.DataFrame,
    test: pd.DataFrame,
    best_iters: list[int],
) -> np.ndarray:
    X = train.drop(columns=["id", "pitnextlap"])
    y = train["pitnextlap"]
    X_test = test.drop(columns=["id"])

    avg_iters = int(np.mean(best_iters))
    print(f"\nTraining on full data with n_estimators={avg_iters}")

    encodings = fit_encodings(X, y)
    X_feat = build_features(X, encodings)[FEATURE_COLS]
    X_test_feat = build_features(X_test, encodings)[FEATURE_COLS]

    params = {k: v for k, v in XGB_PARAMS.items() if k != "early_stopping_rounds"}
    params["n_estimators"] = avg_iters

    model = xgb.XGBClassifier(**params)
    model.fit(X_feat, y, verbose=200)

    return model.predict_proba(X_test_feat)[:, 1]


def save_submission(test: pd.DataFrame, preds: np.ndarray, oof_auc: float) -> Path:
    SUBMISSIONS_DIR.mkdir(exist_ok=True)
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    fname = SUBMISSIONS_DIR / f"submission_xgb_auc{oof_auc:.4f}_{timestamp}.csv"
    pd.DataFrame({"id": test["id"], "PitNextLap": preds}).to_csv(fname, index=False)
    print(f"Saved: {fname}")
    return fname

## Load data

In [ ]:
train, test = load_data()
print(f"Train: {train.shape}  |  Test: {test.shape}")
print(f"Target rate: {train['pitnextlap'].mean():.3f}")
train.head(3)

## Feature engineering — quick sanity check

In [ ]:
X_sample = train.drop(columns=['id', 'pitnextlap']).head(1000)
y_sample = train['pitnextlap'].head(1000)

enc_sample = fit_encodings(X_sample, y_sample)
feat_sample = build_features(X_sample, enc_sample)[FEATURE_COLS]

print(f"Feature matrix shape: {feat_sample.shape}")
print(f"NaN count per column:")
feat_sample.isna().sum()[feat_sample.isna().sum() > 0]

In [ ]:
feat_sample.describe()

## 5-Fold Cross-Validation

In [ ]:
oof_preds, best_iters = run_cv(train)

In [ ]:
oof_auc = roc_auc_score(train['pitnextlap'], oof_preds)
print(f"OOF AUC: {oof_auc:.5f}")
print(f"Best iterations per fold: {best_iters}  |  mean: {np.mean(best_iters):.0f}")

In [ ]:
fpr, tpr, _ = roc_curve(train['pitnextlap'], oof_preds)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'XGBoost OOF (AUC={oof_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('OOF ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
# Train a quick model on a 20% sample to get feature importances
X = train.drop(columns=['id', 'pitnextlap'])
y = train['pitnextlap']

sample_idx = X.sample(frac=0.2, random_state=42).index
X_s, y_s = X.loc[sample_idx], y.loc[sample_idx]

enc = fit_encodings(X_s, y_s)
X_feat = build_features(X_s, enc)[FEATURE_COLS]

params = {k: v for k, v in XGB_PARAMS.items() if k != 'early_stopping_rounds'}
params['n_estimators'] = 300
quick_model = xgb.XGBClassifier(**params)
quick_model.fit(X_feat, y_s, verbose=False)

importance = pd.Series(
    quick_model.feature_importances_, index=FEATURE_COLS
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 10))
importance.plot.barh(ax=ax)
ax.set_title('XGBoost Feature Importance (gain)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## Train on full data & generate submission

In [ ]:
test_preds = train_full_and_predict(train, test, best_iters)

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(test_preds, bins=50, edgecolor='white')
plt.xlabel('Predicted probability')
plt.ylabel('Count')
plt.title('Test prediction distribution')
plt.tight_layout()
plt.show()
print(f"Test pred mean: {test_preds.mean():.4f}  |  Train target mean: {train['pitnextlap'].mean():.4f}")

In [ ]:
submission_path = save_submission(test, test_preds, oof_auc)
print(f"Submission saved: {submission_path}")